In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import os

In [ ]:
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)

ds = load_dataset("ServiceNow/repliqa")
repliqa_4_ds = ds['repliqa_4']

In [ ]:
repliqa_4_ds[0]

In [ ]:
documents = []

for item in repliqa_4_ds:

    # handle missing keys safely
    answer = item.get("answer", "")
    long_answer = item.get("long_answer", "")
    context = item.get("document_extracted", "")
    topic = item.get("document_topic", "")


    if (
        not context or long_answer == "NA" or topic != "Company Policies"
        or answer == "The answer is not found in the document."
    ):
        continue

    documents.append(item)


In [ ]:
len(documents)

In [ ]:
type(documents[0])

In [ ]:
from sklearn.model_selection import train_test_split

train_docs, val_test_docs = train_test_split(documents, test_size=0.4)

In [ ]:
len(train_docs)

In [ ]:
len(val_test_docs)

In [ ]:
val_docs, test_docs = train_test_split(val_test_docs, test_size=0.5)

In [ ]:
len(val_docs)

In [ ]:
len(test_docs)

In [ ]:
from datasets import Dataset, DatasetDict

# Convert list of dictionaries → Dataset
train_dataset = Dataset.from_list(train_docs)
val_dataset = Dataset.from_list(val_docs)
test_dataset = Dataset.from_list(test_docs)

# Create DatasetDict
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# Push to Hugging Face Hub
dataset_dict.push_to_hub("ayanmukherjee/repliqa-4-company-policies-answerable")